In [1]:
import numpy as np 
import pandas as pd

In [2]:
movies=pd.read_csv('final.csv')
movies.shape

(135127, 9)

In [3]:
movies[movies.duplicated(subset='id', keep=False)]
movies.drop_duplicates(subset='id', keep='first', inplace=True)
movies.duplicated().sum()


np.int64(0)

In [4]:
import re

# Clean and lowercase keywords, keep as string
movies['keywords'] = movies['keywords'].fillna('').astype(str).apply(
    lambda x: re.sub(r'[^\w\s]', '', x).lower()
)

# Same for overview
movies['overview'] = movies['overview'].fillna('').astype(str).apply(
    lambda x: re.sub(r'[^\w\s]', '', x).lower()
)

# Same for genres
movies['genres'] = movies['genres'].fillna('').astype(str).apply(
    lambda x: re.sub(r'[^\w\s]', '', x).lower()
)

print("HIHI")

HIHI


In [5]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [6]:
def stem(text):
    l=[]
    for i in text.split():
        l.append(ps.stem(i))
    string=" ".join(l)
    return string
    

In [7]:
title=movies[['id','title']]
# title['title']=title['title'].apply(lambda x: x.split())
overview=movies[['id','overview']]
genres=movies[['id','genres']]
keywords=movies[['id','keywords']]
director=movies[['id','director']]
cast=movies[['id','cast']]


In [8]:
print("HIHI")
genres.loc[:, 'genres'] = genres['genres'].apply(stem)
print("HIHI")
overview.loc[:, 'overview'] = overview['overview'].apply(stem)
print("HIHI")
keywords.loc[:, 'keywords'] = keywords['keywords'].apply(stem)
print("HIHI")

HIHI
HIHI
HIHI
HIHI


In [9]:
director = director.copy()
director['director'] = director['director'].fillna('unknown_director')


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

In [11]:
print("HIHI")
cv_genres=CountVectorizer(stop_words='english')
genres_vectors=cv_genres.fit_transform(genres['genres'])
print("HIHI")

cv_keywords=CountVectorizer(stop_words='english')
keywords_vectors=cv_keywords.fit_transform(keywords['keywords'])
print("HIHI")

cv_overview=CountVectorizer(stop_words='english')
overview_vectors=cv_overview.fit_transform(overview['overview'])
print("HIHI")


HIHI
HIHI
HIHI
HIHI


In [12]:
cv_director = CountVectorizer(stop_words='english')
director_vectors = cv_director.fit_transform(director['director'])
director_vectors = director_vectors.tolil()   #: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
unknown_index=cv_director.vocabulary_.get('unknown_director')

if unknown_index is not None:
    director_vectors=director_vectors.copy()
    director_vectors[:,unknown_index]=0
director_vectors = director_vectors.tocsr()       # Convert back to CSR for fast computations
print("HIHI")

HIHI


In [13]:
cast = cast.dropna(subset=['cast']).reset_index(drop=True)


In [14]:
import ast

# Step 1: Convert cast strings to actual lists of tuples
cast['cast'] = cast['cast'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Step 2: Expand and convert to space-separated string
def expand_cast_to_string(cast_list):
    if not isinstance(cast_list, list):
        return ''
    expanded = [name for name, count in cast_list if isinstance(name, str) and isinstance(count, int) for _ in range(count)]
    return ' '.join(expanded)

# Step 3: Apply the function
cast['cast'] = cast['cast'].apply(expand_cast_to_string)
cast['cast'] = cast['cast'].str.replace('-', '', regex=False)


In [15]:
print(cast.iloc[0]['cast'])


leonardodicaprio leonardodicaprio leonardodicaprio josephgordonlevitt josephgordonlevitt josephgordonlevitt elliotpage elliotpage elliotpage kenwatanabe kenwatanabe kenwatanabe tomhardy tomhardy tomhardy dileeprao cillianmurphy tomberenger


In [16]:

cv_cast=CountVectorizer(stop_words='english')
cast_vectors=cv_cast.fit_transform(cast['cast'])

In [17]:
# Get the feature names (i.e. the vocabulary)
feature_names = cv_cast.get_feature_names_out()

# Convert the first row to a dense array
first_row_vector = cast_vectors[0].toarray()[0]

# Combine feature names with counts
first_row_vocab = dict(zip(feature_names, first_row_vector))

# Show only non-zero entries (i.e. the words present in the row)
{word: count for word, count in first_row_vocab.items() if count > 0}


{'cillianmurphy': np.int64(1),
 'dileeprao': np.int64(1),
 'elliotpage': np.int64(3),
 'josephgordonlevitt': np.int64(3),
 'kenwatanabe': np.int64(3),
 'leonardodicaprio': np.int64(3),
 'tomberenger': np.int64(1),
 'tomhardy': np.int64(3)}

In [18]:
director.shape

(135124, 2)

In [19]:
import numpy as np
import faiss
from sklearn.decomposition import TruncatedSVD

# ---- Helper Functions ----

def maybe_reduce_vector(x, name="vector", n_components=128):
    """Apply TruncatedSVD only if input is high-dimensional (and sparse)."""
    original_dim = x.shape[1]

    if n_components < original_dim:
        print(f"Reducing {name} from {original_dim} → {n_components} dimensions")
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        x_reduced = svd.fit_transform(x)
        return x_reduced.astype(np.float32)
    else:
        print(f"Skipping reduction for {name} (dim = {original_dim})")
        return x.toarray().astype(np.float32)  # safe to convert small ones

def normalize(vectors):
    """Normalize vectors row-wise for cosine similarity."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / (norms + 1e-10)

# ---- STEP 1: Load and Reduce Vectors ----

genres_vectors   = maybe_reduce_vector(genres_vectors, 'genres', 64)
keywords_vectors = maybe_reduce_vector(keywords_vectors, 'keywords', 64)
overview_vectors = maybe_reduce_vector(overview_vectors, 'overview', 128)
cast_vectors     = maybe_reduce_vector(cast_vectors, 'cast', 64)
director_vectors = maybe_reduce_vector(director_vectors, 'director', 32)

# ---- STEP 2: Apply weights ----
genres_vectors   = genres_vectors * 10
keywords_vectors = keywords_vectors * 6
overview_vectors = overview_vectors * 6
# cast_vectors: unweighted
director_vectors = director_vectors * 4

# ---- STEP 3: Combine all vectors horizontally ----
combined_vectors = np.hstack([
    genres_vectors,
    keywords_vectors,
    overview_vectors,
    cast_vectors,
    director_vectors
])

# ---- STEP 4: Normalize for cosine similarity ----
combined_norm = normalize(combined_vectors)

# ---- STEP 5: Build FAISS index ----
index = faiss.IndexFlatIP(combined_norm.shape[1])
index.add(combined_norm)

# ---- STEP 6: Search ----
k = 500
D, I = index.search(combined_norm, k)

# ---- STEP 7: Save ----
np.save('similar_indices.npy', I)
np.save('similar_scores.npy', D)

print("✅ Done. Results saved.")


Skipping reduction for genres (dim = 21)
Reducing keywords from 14274 → 64 dimensions
Reducing overview from 118507 → 128 dimensions
Reducing cast from 308947 → 64 dimensions
Reducing director from 52882 → 32 dimensions


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 135124 and the array at index 3 has size 108667